# Hydrostatic Equilibrium (2D)

This notebook runs the hydrostatic-equilibrium benchmark: a dense square sits in a lighter background at uniform pressure, so the exact solution is "nothing happens" -- every velocity should stay at (or very near) zero for the whole run. What the plot actually shows is the spurious surface tension SPH produces at the density jump, which is the point of the test; `maxVelocity` (the case's own diagnostic, alongside the usual energies) is the number that quantifies it.

This is the first of the `particlePlot`-based (2D field view) notebooks in this backlog, unlike `01-sod`/`03-Kidder`/`04-Noh`/`05-Woodward_Colella`, which all scatter 1D profiles against `x`. The window/event-loop-free core of `hydrostaticCase.setupPlot`/`updatePlot` is exported from `warpSPH.cases.plotting` as `buildFieldPlotter`/`refreshFieldPlotter` for exactly this reason -- called directly below instead of the `Case` hooks, which go through `openWindow`/`pumpEvents` and do not live-update reliably inside a Jupyter cell in this environment (see `sod_1d.ipynb` for the same reasoning applied to the 1D `profilePlot` case). The two panels themselves (`velocities`, `densities`) are `HYDROSTATIC_FIELDS`, exported from `warpSPH.cases.hydrostatic` rather than re-derived here.

`hydrostaticCase` has no `timestep`/`postStep` hook, so `dt` is fixed for the whole run and the loop below is a plain `range(nSteps)`, the same shape as `04-noh-implosion.ipynb`'s.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/08-Hydrostatic.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.hydrostatic import hydrostaticCase, HYDROSTATIC_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `08-hydrostatic.py`, made explicit and editable here.
# `hydrostaticCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=hydrostaticCase.name, scheme=hydrostaticCase.scheme,
                params=dict(hydrostaticCase.params)) \
    .merged(**hydrostaticCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=200,
    dim=2,
    L=1.0,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,
    # buildHydrostaticInitialState leaves dt unset, so this case needs an
    # explicit one -- there is no `timestep` hook, so it stays fixed.
    dt=2.5e-3,

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- hydrostatic's own knobs -----------------------------------------------
    params=dict(
        rho_low=1.0, rho_high=2.0, E0=1.0,
        markerSize=4,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`hydrostaticCase.buildSystem` -> `buildHydrostaticInitialState`), not
# re-derived here.
ctx = buildContext(hydrostaticCase, spec)
hydrostaticCase.configureScheme(ctx)
system = hydrostaticCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(HYDROSTATIC_FIELDS), not hydrostaticCase.setupPlot
# -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, HYDROSTATIC_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = hydrostaticCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=hydrostaticCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = hydrostaticCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, HYDROSTATIC_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=hydrostaticCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
